In [ ]:
!pip install bertopic sentence-transformers umap-learn hdbscan

In [ ]:
import pandas as pd
import os
import numpy as np
import re
import string
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

=== 50 mots les plus fréquents (hors stopwords) ===
  'est: 42
  position: 31
  tête: 28
  finale: 24
  faut: 21
  transition: 21
  temps: 20
  demifinale: 19
  voit: 18
  parti: 18
  passe: 16
  vraiment: 16
  diamants: 15
  oui: 15
  ski: 13
  départ: 12
  marches: 12
  descente: 12
  olympique: 12
  partie: 12
  petit: 11
  n'y: 11
  place: 11
  l'instant: 11
  qu'elle: 11
  monde: 10
  sprint: 10
  skis: 9
  médaille: 9
  zone: 9
  qu'on: 9
  sûr: 9
  podium: 9
  revenir: 8
  revient: 8
  évidemment: 8
  également: 8
  dernière: 7
  difficile: 7
  course: 7
  avantpostes: 7
  compliqué: 7
  russe: 7
  prend: 7
  part: 7
  autant: 7
  alpinisme: 7
  déjà: 7
  vite: 6
  gauche: 6

Calcul des embeddings sur : cpu


/opt/python/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Batches: 100%|██████████| 11/11 [01:17<00:00,  7.05s/it]

Embeddings sauvegardés dans work/PESSD-GBDOC/embeddings.npy
Corpus : 654 phrases | embeddings : (654, 768)

=== Similarité cosinus F vs H par terme ===
 terme  n_F  n_H  similarite_cosinus
effort    2    5               0.788


In [ ]:
# ── 1. CHARGER LE CORPUS ──────────────────────────────────────────────────────

#Chemin à adapter
input_dir = "work/PESSD-GBDOC"

df = pd.DataFrame()
for file in os.listdir(input_dir):
    if file.endswith(".csv"):
        temp = pd.read_csv(os.path.join(input_dir, file))
        df = pd.concat([df, temp])

In [ ]:
# ── 2. NETTOYAGE ──────────────────────────────────────────────────────────────

df = df[(df["main_g"]=="female") | (df["main_g"]=="male")]
df["text"] = df["text"].apply(lambda x: x.split(". "))
df = df.explode("text")
df["text"] = df["text"].apply(lambda x: re.sub(r"\s*[A-Z]\w*\s*", " ", x).strip())
df["text"] = df["text"].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation.replace("'", ""))))
df["g_ath"] = df["audio_file"].apply(lambda x: x.replace(".wav", "").split("-")[-1])
df = df.reset_index(drop=True)


In [ ]:
# ── 3. EXPLORER LE VOCABULAIRE RÉEL DU CORPUS ─────────────────────────────────

STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

words = " ".join(df["text"].tolist()).split()
words_filtered = [w for w in words if w not in STOPWORDS and len(w) > 2]
freq = Counter(words_filtered).most_common(50)

print("=== 50 mots les plus fréquents (hors stopwords) ===")
for word, count in freq:
    print(f"  {word}: {count}")

In [ ]:
# ── 4. CALCULER LES EMBEDDINGS (GPU + cache) ──────────────────────────────────

embeddings_path = "work/PESSD-GBDOC/embeddings.npy"

if os.path.exists(embeddings_path):
    print("\nChargement des embeddings depuis le cache...")
    embeddings = np.load(embeddings_path)
    print(f"Embeddings chargés : {embeddings.shape}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nCalcul des embeddings sur : {device}")

    embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base", device=device)

    embeddings = embedding_model.encode(
        df["text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    np.save(embeddings_path, embeddings)
    print(f"Embeddings sauvegardés dans {embeddings_path}")

df["embedding"] = list(embeddings)
print(f"Corpus : {len(df)} phrases | embeddings : {embeddings.shape}")


In [8]:
# ── 5. DISTANCE COSINUS PAR TERME ─────────────────────────────────────────────

# Adapter cette liste après avoir regardé les 50 mots les plus fréquents
TERMES = ["position", "effort", "finale", "olympique", "tête", "descente", "ski", "marche","médaille","sprint","course"]

results = []

for terme in TERMES:
    mask_F = df["g_ath"].str.upper() == "F"
    mask_H = df["g_ath"].str.upper() == "H"
    mask_terme = df["text"].str.contains(terme, case=False, na=False)

    emb_F = np.array(df[mask_F & mask_terme]["embedding"].tolist())
    emb_H = np.array(df[mask_H & mask_terme]["embedding"].tolist())

    if len(emb_F) == 0 or len(emb_H) == 0:
        results.append({"terme": terme,  "similarite_cosinus": None})
        continue

    centroid_F = emb_F.mean(axis=0, keepdims=True)
    centroid_H = emb_H.mean(axis=0, keepdims=True)

    sim = cosine_similarity(centroid_F, centroid_H)[0][0]
    results.append({"terme": terme, "similarite_cosinus": round(sim, 4)})

# ── 6. RÉSULTATS ──────────────────────────────────────────────────────────────

results_df = pd.DataFrame(results).dropna(subset=["similarite_cosinus"]).sort_values("similarite_cosinus")
print("\n=== Similarité cosinus F vs H par terme ===")
print(results_df.to_string(index=False))


=== Similarité cosinus F vs H par terme ===
    terme  similarite_cosinus
   course              0.4344
   sprint              0.5698
      ski              0.7866
   effort              0.7880
   marche              0.8015
 médaille              0.8099
     tête              0.8175
 descente              0.8317
olympique              0.8557
   finale              0.9127
 position              0.9396
